In [ ]:
import os
import gdown
import shutil
import rasterio
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
source = "/content/drive/MyDrive/satalite data/data"

destination = "/content/SatelliteDataset"

shutil.copytree(source, destination)



In [ ]:
for root , dirs , files in os.walk(destination):
  print(f"Root : {root}")
  print(f"Directories : {dirs}")
  print(f"Files : {files}")
  print(f"No. of Files :{len(files)}")
  print("--------------")

In [ ]:
with rasterio.open("/content/SatelliteDataset/images/0.tif") as src:
    print("Bands:", src.count)
    print("Shape:", src.height, src.width)
    print("Data type:", src.dtypes)
    print("Dscr:", src.descriptions)
    print("CRS:", src.crs)
    print("Transform:", src.transform)


    image = src.read()   # shape: (bands, height, width)


In [ ]:
for i in range(12):
    print(f"Band {i}: min={image[i].min()}, max={image[i].max()}, mean={image[i].mean()}")


In [ ]:
for i , im in enumerate(sorted(os.listdir("/content/SatelliteDataset/images"))):

  path = os.path.join("/content/SatelliteDataset/images" , im)

  with rasterio.open(path) as src:
    image = src.read()

  # Normalize only RGB bands
  rgb = image[[3,2,1]].astype(np.float32)

  for c in range(3):
      band = rgb[c]
      rgb[c] = (band - band.min()) / (band.max() - band.min())

  rgb = np.transpose(rgb, (1, 2, 0))

  #plt.figure(figsize=(5, 5))

  plt.imshow(rgb)
  plt.title(f"RGB Preview {im}")
  plt.show()

  if i == 20:
    break


In [ ]:
def preview_satellite_image_grid(image_path):

    with rasterio.open(image_path) as src:
        image = src.read()  # shape: (12, H, W)

    band_names = [
        "Coastal", "Blue", "Green", "Red",
        "NIR", "SWIR1", "SWIR2",
        "QA Band", "Merit DEM", "Copernicus DEM",
        "ESA World Cover", "Water Occurrence Probability"
    ]

    # Prepare figure
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    axes = axes.flatten()

    rgb = image[[3,2,1]].astype(np.float32)
    for c in range(3):
        band = rgb[c]
        rgb[c] = (band - band.min()) / (band.max() - band.min())
    rgb = np.transpose(rgb, (1,2,0))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB Composite")
    axes[0].axis("off")

    # Indices to show: 0, 4, 5, 6, 7, 8, 9, 10, 11
    channel_indices = [0, 4, 5, 6, 7, 8, 9, 10, 11]

    for ax_idx, i in enumerate(channel_indices, start=1):
        band = image[i].astype(np.float32)
        name = band_names[i]

        # Choose colormap and normalization
        if i <= 6:  # spectral bands
            band = (band - band.min()) / (band.max() - band.min())
            cmap = "Blues"
        elif i in [8,9]:  # DEMs
            band = (band - band.min()) / (band.max() - band.min())
            cmap = "Blues"
        elif i == 11:  # water probability
            band = band / 100.0
            cmap = "Blues"
        else:  # QA / World Cover
            cmap = "Blues"

        axes[ax_idx].imshow(band, cmap=cmap)
        axes[ax_idx].set_title(name)
        axes[ax_idx].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
folder = "/content/SatelliteDataset/images"
files = sorted([f for f in os.listdir(folder) if f.endswith(".tif")])

# Preview first 3 images
for f in files[:20]:
  print(f"Image : {f}")
  preview_satellite_image_grid(os.path.join(folder, f))
  print("\n")


In [ ]:
input_data = sorted(os.listdir("/content/SatelliteDataset/images"))
labels_data = sorted(os.listdir("/content/SatelliteDataset/labels"))


for i in range(len(max(input_data , labels_data))):

  print(f"Trainging : {input_data[i]} ---- Label : {labels_data[i]}")

In [ ]:
label_preview = ["113.tif" , "113.png" ,"113_44.png"]



for i in label_preview:
    if i.endswith(".tif"):
        path = os.path.join("/content/SatelliteDataset/images" , i)

        with rasterio.open(path) as src:
            image = src.read()

        # Normalize only RGB bands
        rgb = image[[3,2,1]].astype(np.float32)

        for c in range(3):
            band = rgb[c]
            rgb[c] = (band - band.min()) / (band.max() - band.min())

        rgb = np.transpose(rgb, (1, 2, 0))

        plt.imshow(rgb)
        plt.title(f"RGB Preview {i}")
        plt.show()
    else:
        path = os.path.join("/content/SatelliteDataset/labels" , i)

        # To view binary masks correctly, use gray colormap
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img * 255
        plt.imshow(img, cmap='gray')
        plt.title(f"RGB Preview {i}")
        plt.show()


In [ ]:
input_data = sorted(os.listdir("/content/SatelliteDataset/images"))
labels_data = sorted(os.listdir("/content/SatelliteDataset/labels"))

filtered_labels = []

for i in os.listdir("/content/SatelliteDataset/labels"):
  if not("_" in i):
    filtered_labels.append(i)
    print(i)


In [ ]:
filtered_labels = sorted(filtered_labels)

for i in range(len(max(input_data , filtered_labels))):

  print(f"Trainging : {input_data[i]} ---- Label : {filtered_labels[i]}")


In [ ]:
num_rows = max(len(filtered_labels), len(input_data))

fig, axes = plt.subplots(num_rows, 2, figsize=(10, 5 * num_rows), squeeze=False)
for i in range(num_rows):
  input_tif = input_data[i]
  input_png = filtered_labels[i]

  path_input = os.path.join("/content/SatelliteDataset/images" , input_tif)
  with rasterio.open(path_input) as src:
      image = src.read()
  # Normalize only RGB bands
  rgb = image[[3,2,1]].astype(np.float32)
  for c in range(3):
      band = rgb[c]
      rgb[c] = (band - band.min()) / (band.max() - band.min())
  rgb = np.transpose(rgb, (1, 2, 0))
  axes[i][0].imshow(rgb)
  axes[i][0].set_title(f"RGB Preview {i}")



  path_png = os.path.join("/content/SatelliteDataset/labels" , input_png)

  # To view binary masks correctly, use gray colormap
  img = cv2.imread(path_png)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img = img * 255
  axes[i][1].imshow(img, cmap='gray')
  axes[i][1].set_title(f"RGB Preview {i}")

plt.tight_layout()
plt.show()

